# Allen MERFISH: Slc6a2 (Norepinephrine Transporter) Across Mouse Brain

This Colab notebook downloads Allen Brain Cell Atlas MERFISH data, extracts expression for **Slc6a2** (NET), and plots its spatial distribution across brain sections.

## 1) Install dependencies (Colab)
Run this cell once per fresh Colab runtime.

In [ ]:
!pip -q install abc-atlas-access anndata pandas numpy matplotlib

## 2) Imports and cache setup
`cache_dir` is where downloaded Allen data will be stored in Colab.

In [ ]:
from pathlib import Path

import anndata
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from abc_atlas_access.abc_atlas_cache.abc_project_cache import AbcProjectCache

plt.rcParams['figure.dpi'] = 120

cache_dir = Path('/content/abc_atlas_cache')
cache_dir.mkdir(parents=True, exist_ok=True)

abc_cache = AbcProjectCache.from_cache_dir(cache_dir)
print('Current manifest:', abc_cache.current_manifest)

## 3) Load MERFISH cell metadata and gene table
Dataset: `MERFISH-C57BL6J-638850` (whole mouse brain MERFISH).

In [ ]:
dataset = 'MERFISH-C57BL6J-638850'

cell = abc_cache.get_metadata_dataframe(
    directory=dataset,
    file_name='cell_metadata_with_cluster_annotation',
    dtype={'cell_label': str, 'neurotransmitter': str}
).set_index('cell_label')

gene = abc_cache.get_metadata_dataframe(
    directory=dataset,
    file_name='gene'
).set_index('gene_identifier')

print('Cells:', f'{len(cell):,}')
print('Genes in MERFISH panel:', len(gene))
cell[['brain_section_label', 'x', 'y']].head()

## 4) Extract Slc6a2 expression from the log2 expression matrix

In [ ]:
target_gene = 'Slc6a2'

match = gene[gene['gene_symbol'] == target_gene]
if match.empty:
    raise ValueError(f'{target_gene} not found in MERFISH gene table.')
if len(match) > 1:
    print('Multiple entries found; using first match.')

gene_id = match.index[0]
print('Gene identifier:', gene_id)

expr_h5ad_path = abc_cache.get_file_path(
    directory=dataset,
    file_name='C57BL6J-638850/log2'
)
print('Expression file:', expr_h5ad_path)

adata = anndata.read_h5ad(expr_h5ad_path, backed='r')
slc6a2 = adata[:, [gene_id]].to_df().rename(columns={gene_id: target_gene})

print(slc6a2.describe().T)
slc6a2.head()

## 5) Join metadata + expression and summarize by section

In [ ]:
joined = cell[['brain_section_label', 'x', 'y']].join(slc6a2, how='inner')

section_mean = (
    joined.groupby('brain_section_label')[target_gene]
    .mean()
    .sort_values(ascending=False)
)

print('Top sections by mean Slc6a2 expression:')
section_mean.head(12)

## 6) Plot Slc6a2 spatial distribution across high-expression sections
This picks the top 4 sections by mean expression and plots all cells in each section, colored by Slc6a2 level.

In [ ]:
top_sections = section_mean.head(4).index.tolist()
print('Sections plotted:', top_sections)

fig, axes = plt.subplots(1, len(top_sections), figsize=(18, 4.8), constrained_layout=True)

if len(top_sections) == 1:
    axes = [axes]

for ax, section in zip(axes, top_sections):
    sub = joined[joined['brain_section_label'] == section]

    # Robust upper clip improves contrast when expression has a long tail.
    vmax = np.percentile(sub[target_gene], 99.5)
    sc = ax.scatter(
        sub['x'],
        sub['y'],
        c=sub[target_gene],
        s=0.2,
        marker='.',
        cmap='magma_r',
        vmin=0,
        vmax=vmax if vmax > 0 else None,
        linewidths=0,
    )

    ax.set_title(section, fontsize=9)
    ax.set_xlim(0, 11)
    ax.set_ylim(11, 0)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_aspect('equal')

cbar = fig.colorbar(sc, ax=axes, fraction=0.02, pad=0.01)
cbar.set_label(f'{target_gene} log2 expression')
plt.suptitle('Spatial distribution of Slc6a2 across selected mouse brain sections', y=1.03)
plt.show()

## 7) Optional: view a broader panel of sections
Increase or decrease `n_sections` depending on runtime/memory.

In [ ]:
n_sections = 12
sections = section_mean.head(n_sections).index.tolist()

n_cols = 4
n_rows = int(np.ceil(len(sections) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 3.6 * n_rows), constrained_layout=True)
axes = np.array(axes).reshape(-1)

for i, section in enumerate(sections):
    ax = axes[i]
    sub = joined[joined['brain_section_label'] == section]
    vmax = np.percentile(sub[target_gene], 99.5)

    sc = ax.scatter(
        sub['x'],
        sub['y'],
        c=sub[target_gene],
        s=0.15,
        marker='.',
        cmap='magma_r',
        vmin=0,
        vmax=vmax if vmax > 0 else None,
        linewidths=0,
    )
    ax.set_title(section, fontsize=8)
    ax.set_xlim(0, 11)
    ax.set_ylim(11, 0)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_aspect('equal')

for j in range(len(sections), len(axes)):
    axes[j].axis('off')

fig.colorbar(sc, ax=axes.tolist(), fraction=0.015, pad=0.01, label=f'{target_gene} log2 expression')
plt.suptitle(f'Top {n_sections} sections by mean Slc6a2 expression', y=1.02)
plt.show()